# Ed25519 Signer
> Signs a canonical N-Quads string (URDNA2015) with **Ed25519** and verifies it.

In [ ]:
#| default_exp verify.signer

## imports

In [ ]:
#| export
import base64
from typing import Tuple

from cogitarelink.core.debug import get_logger
log = get_logger("signer")

try:
    from nacl.signing import SigningKey, VerifyKey           # type: ignore
    from nacl.encoding import RawEncoder
    _HAS_NACL = True
except ModuleNotFoundError:
    _HAS_NACL = False
    log.warning("PyNaCl not installed â†’ signer disabled")

## helpers

In [ ]:
#| export
def _b64(b: bytes) -> str: return base64.b64encode(b).decode()
def _unb64(s: str) -> bytes: return base64.b64decode(s.encode())

## public API

In [ ]:
#| export
def generate_keypair() -> Tuple[str, str]:
    """
    Returns `(public_b64, private_b64)`.

    Raises RuntimeError if PyNaCl missing.
    """
    if not _HAS_NACL:
        raise RuntimeError("sign/verify require PyNaCl (pip install pynacl)")
    sk = SigningKey.generate()
    return _b64(sk.verify_key.encode()), _b64(sk.encode())

def sign(normalized_nquads: str, private_key_b64: str) -> str:
    if not _HAS_NACL:
        raise RuntimeError("PyNaCl missing")
    sk = SigningKey(_unb64(private_key_b64), encoder=RawEncoder)
    sig = sk.sign(normalized_nquads.encode()).signature
    return _b64(sig)

def verify(normalized_nquads: str, signature_b64: str, public_key_b64: str) -> bool:
    if not _HAS_NACL:
        raise RuntimeError("PyNaCl missing")
    vk = VerifyKey(_unb64(public_key_b64), encoder=RawEncoder)
    try:
        vk.verify(normalized_nquads.encode(), _unb64(signature_b64))
        return True
    except Exception:
        return False

## quick tests

In [ ]:
#| hide
if _HAS_NACL:
    pub, priv = generate_keypair()
    msg = "<x> <y> \"z\" .\n"
    sig = sign(msg, priv)
    assert verify(msg, sig, pub)
    assert verify(msg + "tamper", sig, pub) is False

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()